<a href="https://colab.research.google.com/github/nabillahumi/S5-Mesin_Learning/blob/main/Jobsheet6%20(ANN)%20PBL/Praktikum%203%20(JS6)%20PBL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Langkah 1: Instalasi hnswlib & Persiapan Dataset

In [2]:
# Instalasi hnswlib (jalankan di sel terpisah jika di Google Colab)
!pip install hnswlib

import numpy as np
import pandas as pd
import time
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# 1. Load dan gabungkan dataset cafes.csv & reviews.csv
cafes = pd.read_csv('cafes.csv')
reviews = pd.read_csv('reviews.csv')

reviews_agg = reviews.groupby('cafe_id').agg(
    avg_review_rating=('rating', 'mean'),
    total_likes=('likes', 'sum')
).reset_index()

merged = pd.merge(cafes, reviews_agg, on='cafe_id', how='left').fillna(0)
print(f"Dataset berhasil dimuat! Jumlah baris data cafe: {len(merged)}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679798 sha256=9e55959057a55326f3d461ef97bed7ef4b50b851edef03f8397a0143cb92e65d
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib
Dataset berhasil dimuat! Jumlah baris data cafe: 99


#Langkah 2: Kode Eksperimen Sesuai Instruksi (2D vs 5D, 1000 Data Asli/Diduplikasi, Metric L2 vs Cosine)

In [3]:
def run_hnsw_experiment(n_target, dim_type):
    # Siapkan data dasar dari merged cafe
    if dim_type == '2D':
        raw_data = merged[['latitude', 'longitude']].values.astype(np.float32)
    else: # 5D
        cols = ['latitude', 'longitude', 'rating', 'review_count', 'avg_review_rating']
        scaler = StandardScaler()
        raw_data = scaler.fit_transform(merged[cols]).astype(np.float32)

    # Scaling jumlah data (jika target 1,000 atau 1,000,000 dengan duplikasi random sampling)
    np.random.seed(42)
    if n_target > len(raw_data):
        indices = np.random.choice(len(raw_data), n_target, replace=True)
        data = raw_data[indices]
    else:
        data = raw_data[:n_target]

    dim = data.shape[1]
    query = data[0:1] # Ambil cafe pertama sebagai query
    k = 5

    results = []
    for metric_name in ['l2', 'cosine']:
        # Sklearn NearestNeighbors (Exact NN)
        sk_metric = 'euclidean' if metric_name == 'l2' else 'cosine'
        nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric=sk_metric)
        nn.fit(data)

        t0 = time.time()
        distances_exact, indices_exact = nn.kneighbors(query)
        time_exact = (time.time() - t0) * 1000 # dalam milidetik

        # HNSW Index
        p = hnswlib.Index(space=metric_name, dim=dim)
        p.init_index(max_elements=n_target, ef_construction=100, M=16)
        p.add_items(data)
        p.set_ef(50)

        t0 = time.time()
        labels_hnsw, distances_hnsw = p.knn_query(query, k=k)
        time_hnsw = (time.time() - t0) * 1000 # dalam milidetik

        results.append({
            "Distance Metrics": metric_name.upper(),
            "Dimensi": dim_type,
            "Jumlah Data": n_target,
            "Hasil Index (Exact vs HNSW)": f"{indices_exact[0].tolist()} vs {labels_hnsw[0].tolist()}",
            "Waktu Komputasi (ms)": f"{time_exact:.4f} vs {time_hnsw:.4f}"
        })
    return results

# Jalankan eksperimen untuk 1,000 data (2D dan 5D)
exp_results = []
exp_results.extend(run_hnsw_experiment(1000, '2D'))
exp_results.extend(run_hnsw_experiment(1000, '5D'))

# (Opsional untuk skala 1 Juta data - jalankan jika RAM memadai)
# exp_results.extend(run_hnsw_experiment(1000000, '2D'))
# exp_results.extend(run_hnsw_experiment(1000000, '5D'))

df_hnsw = pd.DataFrame(exp_results)
display(df_hnsw)

,Distance Metrics,Dimensi,Jumlah Data,Hasil Index (Exact vs HNSW),Waktu Komputasi (ms)
0,L2,2D,1000,"[213, 210, 0, 267, 158] vs [210, 213, 267, 445...",85.4645 vs 0.0730
1,COSINE,2D,1000,"[664, 649, 650, 651, 652] vs [58, 115, 124, 15...",13.1748 vs 0.0825
2,L2,5D,1000,"[210, 158, 0, 267, 213] vs [210, 267, 464, 948...",0.6304 vs 0.0770
3,COSINE,5D,1000,"[0, 998, 623, 866, 158] vs [0, 210, 267, 866, ...",13.3924 vs 0.0827
